# Siparişler - `review_score` Çok Değişkenli Regresyonu

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Import modules
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

⚠️ Devam etmeden önce:
* 💾 Önceki ünitenin Siparişler çözümünü indirin
* 👥 `order_solution.py`'nin içeriğini `olist/order.py` dosyasına kopyalayıp yapıştırın

> ✅ **Not (Poi):** `olist/order.py` zaten tamam ve commit'li — bu adımı atlıyoruz.

⚠️ `olist` havuzunuzda `order.py` dosyasındaki kod değişikliklerini commit etmeyi unutmayın!

👇 Öncelikle `orders` veri setini içeri aktarın

In [ ]:
from olist.order import Order
orders = Order().get_training_data(with_distance_seller_customer=True)
orders.head()

Önceki analizimizi hatırlayalım:

Korelasyon matrisinde `review_score` çoğunlukla `wait_time` ve `delay_vs_expected` ile ilişkili. Ancak bu iki özellik birbirleriyle de yüksek ilişkili.

Bu alıştırmada `statsmodels` ile bir özelliğin etkisini, **diğerlerini sabit tutarak** ayırt edeceğiz.

In [ ]:
plt.figure(figsize = (10, 10))

sns.heatmap(
    orders.corr(numeric_only=True),
    cmap='coolwarm',
    annot=True,
    annot_kws={"size": 10}
);

## 1 - Tek Değişkenli Regresyon

❓ `statsmodels.formula.api` ile şunları oluşturun:
 - `model1`: `review_score` ~ `wait_time`
 - `model2`: `review_score` ~ `delay_vs_expected`

Her biri için `summary` yazdırın: `R-squared`, katsayılar, `t`, `p` ve `%95 güven aralıkları`.

***Model 1***:

In [ ]:
model1 = smf.ols(formula='review_score ~ wait_time', data=orders).fit()
model1.summary()

***Model 2***:

In [ ]:
model2 = smf.ols(formula='review_score ~ delay_vs_expected', data=orders).fit()
model2.summary()

## 2 - Çok Değişkenli Regresyon

❓ `wait_time` sabit tutulduğunda, bir gün `delay_vs_expected` eklemenin `review_score` üzerindeki etkisi nedir? Hangisi düşük puanı daha çok açıklar?

`wait_time` ve `delay_vs_expected`'i bağımsız, `review_score`'u hedef alan `model3`'ü çalıştırın.

***Model 3***:

In [ ]:
model3 = smf.ols(formula='review_score ~ wait_time + delay_vs_expected', data=orders).fit()
model3.summary()

----
👉 Çok değişkenli regresyon bir özelliğin etkisini izole eder; bu katsayılara **kısmi korelasyon katsayıları** denir.

❓ *seaborn* basit regresyon katsayılarıyla fark var mı? `wait_time` ve `delay_vs_expected` göreceli eğimleri hakkında ne söylenir?

<details>
    <summary>- 💡 Solution 💡-</summary>

- Holding `wait_time` constant, each additional day of `delay` reduces the review_score on average by 0.0205 [0.023 - 0.018] points
- Holding `delay` constant, each additional day of `wait_time` reduces the review_score on average by 0.0383 [0.039 - 0.037] points

Contrary to the simple bivariate analysis, `delay` is actually less impactful than `wait_time` in driving lower `review_score`! Multi-variate regression removes the impact of confounding factors.

> 📝 **Yorum (Poi):**
> Basit (seaborn) regresyonda `delay` eğimi (≈ −0.10) `wait_time`'dan (≈ −0.05) dik görünüyordu.
> Çok değişkenli `model3`'te tablo **tersine dönüyor**: `wait_time` (≈ **−0.038**) `delay`'den (≈ **−0.020**) daha güçlü.
> Neden? İki değişken birbiriyle yüksek korelasyonlu — basit regresyonda `delay`, `wait_time`'ın etkisini de üstleniyordu.
> İkisini aynı modele koyunca etki **paylaşılıyor** ve gerçek sürücünün toplam bekleme süresi olduğu ortaya çıkıyor.
> Bu, tam olarak görevin uyardığı **karıştırıcı (confounding)** etkisinin sayısal kanıtı.

---
❌ R-squared düşük: review_score varyasyonunun en fazla ~%12'si bu iki özellikle açıklanıyor.

✅ Açıklanabilirliği artırmak için daha fazla özellik ekleyelim (`model4`).

📝 <u>Not</u>: **Çok Değişkenli Doğrusal Regresyon** = **Ordinary Least Squares**; **MSE**'yi minimize eder.

***Model 4***:

❓ Hangi özellikleri seçersiniz? Bir `features` DataFrame oluşturun.

- ⚠️ **veri sızıntısı** yok: `review_score`'tan türetilen özellikleri ekleme (`dim_is_five_star`, `dim_is_one_star`)
- ⚠️ Birbiriyle mükemmel korelasyonlu iki özellik ekleme

In [ ]:
# Feature set: drop the target and anything derived from it (leakage),
# plus the non-numeric id/status columns.
features = orders.drop(columns=[
    'review_score',        # the target
    'dim_is_five_star',    # derived from review_score -> leakage
    'dim_is_one_star',     # derived from review_score -> leakage
    'order_id',            # identifier, not a feature
    'order_status'         # constant ('delivered') after filtering
])
features.head()

Sonra özellikleri **standardize** edeceğiz (z-score $Z_i = \frac{X_i - \mu_i}{\sigma_i}$).

**Neden?** Ölçek farkları yüzünden bazı özellikler yapay olarak daha önemli görünür. Katsayıları karşılaştırılabilir yapmak için aynı ölçeğe getiririz.

👉 Adımlar: `features`'tan başla → `mean` çıkar → `std`'ye böl → `orders_standardized`'e kaydet → `review_score`'u ekle.

In [ ]:
# z-score each feature. NOTE: parentheses matter -
# (X - mean) / std, NOT X - mean/std (the hint's precedence is wrong)
orders_standardized = (features - features.mean()) / features.std()

# Add the (un-standardized) target back for the formula
orders_standardized['review_score'] = orders['review_score']
orders_standardized.head()

👉 `model4`'ü oluşturun ve eğitin.

In [ ]:
# Build the formula: review_score ~ all standardized features
feature_names = [c for c in orders_standardized.columns if c != 'review_score']
formula = 'review_score ~ ' + ' + '.join(feature_names)

model4 = smf.ols(formula=formula, data=orders_standardized).fit()
model4.summary()

---
❓ En önemli özellikler? (`.plot(kind='barh')` ile çubuk grafik)
- Genel performans nasıl değişti?
- Regresyon istatistiksel olarak anlamlı mı?

In [ ]:
# Drop the intercept; sort coefficients by magnitude for a clean ranking
coefs = model4.params.drop('Intercept').sort_values()

coefs.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Standardized coefficients (impact on review_score)')
plt.xlabel('coefficient (z-score units)')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"R-squared: {model4.rsquared:.4f}")
print(f"F p-value: {model4.f_pvalue:.2e}")

<details>
    <summary>- 💡 Açıklamalar 💡 -</summary>

- `wait_time` en büyük açıklayıcı değişkendir
- Bir siparişte ne kadar çok `items` ve `sellers` varsa `review_score` o kadar düşük
- Mesafe de rol oynar
- `price` / `freight_value` p-değerleri çok yüksekse sonuç çıkarılamayabilir

- F-istatistiği 1'den çok büyük → regresyon istatistiksel olarak anlamlı
- R-squared çok artmadı → açıklamanın çoğu orders veri seti dışında

⚠️ n >> p olduğunda düşük R-squared yaygındır; anlamlı olduğu sürece içgörü yine de geçerlidir.
</details>

## 3 - Model Performansını Kontrol Edin

⚠️ Performans sadece R-squared ile ölçülmez!

👀 Tahminlerin ve özellikle **kalıntıların** dağılımını her zaman görselleştirin.

❓ Kalıntıları hesaplayın. Ortalamaları ≈ 0 olmalı.

In [ ]:
# Residual = actual - predicted
predicted_review_score = model4.predict(orders_standardized)
residuals = orders_standardized['review_score'] - predicted_review_score

print(f"Mean of residuals: {residuals.mean():.2e}")   # ~0, as expected for OLS

🧮 İlişkili RMSE'yi hesaplayın.

In [ ]:
# RMSE = sqrt(mean(residual^2))
rmse = np.sqrt((residuals ** 2).mean())
print(f"RMSE: {rmse:.4f}")

📊 `residuals`'ı bir histogramda çizin.

In [ ]:
sns.histplot(residuals, bins=50)
plt.title('Distribution of residuals')
plt.xlabel('residual (actual - predicted review_score)')
plt.show()

❓ Kalıntıların şekli neden garip?

*İpucu:* Aynı grafikte `review_score` ile `predicted_review_score` dağılımlarını çizin.

In [ ]:
# Actual is 5 discrete spikes (1-5); predicted is a narrow continuous blob.
# Their difference => several overlapping shifted humps => the 'weird' shape.
sns.histplot(orders_standardized['review_score'], bins=50,
             color='steelblue', label='actual review_score', stat='density')
sns.histplot(predicted_review_score, bins=50,
             color='orange', label='predicted review_score', stat='density')
plt.legend()
plt.title('Actual vs predicted review_score')
plt.show()

📈 Önceki challenge'ın regresyon çizgisini yeniden çizelim:

In [ ]:
sample = orders.sample(10000, random_state=42)
plt.figure(figsize=(13,5))
plt.suptitle('Regression of review_score, 95% confidence interval')
plt.subplot(1,2,1)
sns.regplot(x = sample.wait_time, y= sample.review_score, y_jitter=.1, ci=95)
plt.xlim(right=70)
plt.ylim(bottom=0)

plt.subplot(1,2,2)
sns.regplot(x = orders.delay_vs_expected, y= orders.review_score, y_jitter=.1, ci=95)
plt.xlim(right=70)
plt.ylim(bottom=0)

☝️ `review_score` ayrı bir sayı (1–5) ve aynı zamanda kategori olduğu için doğrusal regresyon zorlanıyor.

📅 Sonraki ünite: adına rağmen bir **sınıflandırma** algoritması olan **Logistic Regression**.

☝️ Sonuç: model iki nedenle zayıf — (1) yeterli özellik yok (düşük R²), (2) doğrusal regresyonu ayrık bir sınıflandırma problemine uyduruyoruz.

💡 Sonraki challenge'da analizi **satıcı seviyesine** toplayacağız.

🏁 Harika iş!

💾 Notebook'u *kaydet*, *commit* ve *push* etmeyi unutma!